# 12: PowerPoint Generation Testing

This notebook tests PowerPoint generation features:

1. **Basic Presentation** - Title, content, metrics slides
2. **Analytics Presentation** - Charts, tables, insights
3. **DataFrame Presentation** - Auto-generated from pandas data
4. **Custom Presentation** - Full control over slide structure

For a real-world example using these capabilities to generate a branded enterprise onboarding deck (PPT + PDF) with charts, tables, and a convergence diagram, see **[Notebook 21: Enterprise Onboarding Presentation](./21_Enterprise_Onboarding_Presentation.ipynb)**.

## Prerequisites

```bash
pip install python-pptx
```

## What this shows
Assembling `Argument` objects into branded PPTX slides via the SAL-66 pipeline.

## Why it matters
Reports/decks now flow from `Chain` → `Argument` → Slides — shared primitive across PDF and PPTX.

## Prereqs
- `pip install 'siege-utilities[reporting,pptx]'`

## Next
- `18_Google_Workspace.ipynb` for Google Slides output; `28_Polling_Survey_Analysis.ipynb` for Argument sources.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import display

# Default guard — cell-3 overrides to True if python-pptx is available
PPTX_AVAILABLE = False

# Ensure siege_utilities is importable
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

print(f"Python: {sys.version}")
print("PowerPoint Generation notebook ready.")

In [ ]:
# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_12"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Check python-pptx availability
try:
    from pptx import Presentation
    from pptx.util import Inches, Pt
    print("python-pptx: available")
    PPTX_AVAILABLE = True
except ImportError as e:
    print(f"python-pptx: NOT available ({e})")
    print("Install with: pip install python-pptx")
    PPTX_AVAILABLE = False

In [ ]:
if not PPTX_AVAILABLE:
    print("WARNING: Skipping remaining cells — python-pptx not installed")
else:
    from siege_utilities.reporting.powerpoint_generator import PowerPointGenerator
    print("PowerPointGenerator imported successfully.")

## 1. Create Sample Data

In [ ]:
if PPTX_AVAILABLE:
    np.random.seed(42)
    
    performance_data = pd.DataFrame({
        'Quarter': ['Q1 2025', 'Q2 2025', 'Q3 2025', 'Q4 2025'],
        'Revenue ($M)': [12.5, 15.2, 18.7, 22.1],
        'Users (K)': [145, 189, 234, 278],
        'Conversion (%)': [3.2, 3.5, 4.1, 4.3]
    })
    
    regional_data = pd.DataFrame({
        'Region': ['Northeast', 'Southeast', 'Midwest', 'West', 'International'],
        'Revenue ($M)': [18.2, 12.5, 8.7, 22.4, 6.7],
        'Growth (%)': [12, 8, 15, 22, 35]
    })
    
    print("Performance Data:")
    display(performance_data)
    print("\nRegional Data:")
    display(regional_data)

## 2. Basic Analytics Presentation

In [ ]:
if PPTX_AVAILABLE:
    pptx_gen = PowerPointGenerator(
        client_name="TestClient",
        output_dir=OUTPUT_DIR
    )
    print(f"PowerPointGenerator initialized (client: {pptx_gen.client_name})")

In [ ]:
if PPTX_AVAILABLE:
    report_data = {
        'executive_summary': """
FY 2025 Performance Highlights:

- Total revenue of $68.5M (+15.4% YoY)
- 846 new customers acquired
- Conversion rate improved to 4.3%
- West region leads with 33% market share
        """,
        'metrics': {
            'Total Revenue': {'value': '$68.5M', 'status': '+15.4%'},
            'New Customers': {'value': '846', 'status': '+23%'},
            'Conversion Rate': {'value': '4.3%', 'status': '+0.8pp'},
            'Customer Satisfaction': {'value': '4.7/5', 'status': 'Excellent'},
            'Net Promoter Score': {'value': '72', 'status': '+5'},
            'Churn Rate': {'value': '2.1%', 'status': '-0.3pp'}
        },
        'charts': [
            {'title': 'Revenue by Quarter', 'type': 'bar'},
            {'title': 'Regional Distribution', 'type': 'pie'}
        ],
        'tables': [
            {
                'title': 'Quarterly Performance',
                'headers': performance_data.columns.tolist(),
                'data': performance_data.values.tolist()
            }
        ],
        'insights': [
            'West region shows highest growth potential',
            'Q3 conversion spike correlates with product launch',
            'International expansion driving new customer growth',
            'Customer satisfaction at all-time high'
        ]
    }
    
    print(f"Report data: {len(report_data['metrics'])} metrics, {len(report_data['insights'])} insights")

In [ ]:
if PPTX_AVAILABLE:
    try:
        analytics_pptx = pptx_gen.create_analytics_presentation(
            report_data=report_data,
            presentation_title="FY 2025 Analytics Report",
            include_charts=True,
            include_tables=True
        )
        
        if analytics_pptx.exists():
            size_kb = analytics_pptx.stat().st_size / 1024
            print(f"Analytics presentation generated: {analytics_pptx.name} ({size_kb:.1f} KB)")
        else:
            print("WARNING: Presentation was not created")
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()

## 3. Performance Presentation

In [ ]:
if PPTX_AVAILABLE:
    perf_data = {
        'overview': """
Performance Overview for FY 2025:

The company exceeded all key performance targets 
with strong growth across all regions.
        """,
        'revenue': {'current': '$68.5M', 'target': '$65.0M', 'achievement': '105%'},
        'customers': {'current': '8,460', 'target': '7,500', 'achievement': '113%'},
        'satisfaction': {'current': '4.7', 'target': '4.5', 'achievement': '104%'},
        'trends': {
            'Revenue': 'Upward trajectory',
            'Customer Growth': 'Accelerating'
        },
        'recommendations': [
            'Increase investment in West region',
            'Launch customer loyalty program',
            'Expand product line for enterprise segment',
            'Accelerate international expansion'
        ]
    }
    
    metrics_list = ['revenue', 'customers', 'satisfaction']
    print("Performance data prepared.")

In [ ]:
if PPTX_AVAILABLE:
    try:
        perf_pptx = pptx_gen.create_performance_presentation(
            performance_data=perf_data,
            metrics=metrics_list,
            presentation_title="FY 2025 Performance Review"
        )
        
        if perf_pptx.exists():
            size_kb = perf_pptx.stat().st_size / 1024
            print(f"Performance presentation generated: {perf_pptx.name} ({size_kb:.1f} KB)")
        else:
            print("WARNING: Presentation was not created")
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()

## 4. DataFrame Presentation

In [ ]:
if PPTX_AVAILABLE:
    sample_df = pd.DataFrame({
        'Product': [f'Product_{i}' for i in range(1, 21)],
        'Sales': np.random.randint(1000, 10000, 20),
        'Revenue': np.random.uniform(10000, 100000, 20).round(2),
        'Growth': np.random.uniform(-10, 30, 20).round(1),
        'Rating': np.random.uniform(3.5, 5.0, 20).round(1)
    })
    
    print(f"Sample DataFrame: {len(sample_df)} rows, {len(sample_df.columns)} columns")
    display(sample_df.head(10))

In [ ]:
if PPTX_AVAILABLE:
    try:
        df_pptx = pptx_gen.create_presentation_from_dataframe(
            df=sample_df,
            presentation_title="Product Performance Analysis",
            max_slides=8
        )
        
        if df_pptx.exists():
            size_kb = df_pptx.stat().st_size / 1024
            print(f"DataFrame presentation generated: {df_pptx.name} ({size_kb:.1f} KB)")
        else:
            print("WARNING: Presentation was not created")
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()

## 5. Custom Presentation with Full Control

In [ ]:
if PPTX_AVAILABLE:
    custom_config = {
        'type': 'quarterly_review',
        'title': 'Q4 2025 Business Review',
        'slides': [
            {
                'type': 'content', 'title': 'Agenda',
                'content': '1. Executive Summary\n2. Financial Performance\n3. Market Analysis\n4. Strategic Initiatives\n5. Q1 2026 Outlook'
            },
            {
                'type': 'content', 'title': 'Executive Summary',
                'content': 'Q4 2025 was our strongest quarter:\n\n- Revenue: $22.1M (+18% QoQ)\n- New customers: 278\n- Market share: 33% in West\n- Customer NPS: 72'
            },
            {
                'type': 'table', 'title': 'Regional Performance',
                'headers': regional_data.columns.tolist(),
                'data': regional_data.values.tolist()
            },
            {
                'type': 'comparison', 'title': 'YoY Comparison',
                'left_content': 'Q4 2024:\n- Revenue: $18.7M\n- Customers: 6,200\n- NPS: 67',
                'right_content': 'Q4 2025:\n- Revenue: $22.1M\n- Customers: 8,460\n- NPS: 72'
            },
            {
                'type': 'content', 'title': 'Next Steps',
                'content': 'Q1 2026 Priorities:\n\n1. Launch enterprise product tier\n2. Expand to European markets\n3. Increase marketing spend 20%\n4. Hire 15 new engineers'
            }
        ]
    }
    
    print(f"Custom presentation config: {len(custom_config['slides'])} slides")

In [ ]:
if PPTX_AVAILABLE:
    try:
        custom_pptx = pptx_gen.create_custom_presentation(custom_config)
        
        if custom_pptx.exists():
            size_kb = custom_pptx.stat().st_size / 1024
            print(f"Custom presentation generated: {custom_pptx.name} ({size_kb:.1f} KB)")
        else:
            print("WARNING: Presentation was not created")
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()

## Summary

In [ ]:
if PPTX_AVAILABLE:
    print("=" * 50)
    print("POWERPOINT GENERATION TEST RESULTS")
    print("=" * 50)
    
    generated_files = [
        ('Analytics PPTX', analytics_pptx),
        ('Performance PPTX', perf_pptx),
        ('DataFrame PPTX', df_pptx),
        ('Custom PPTX', custom_pptx),
    ]
    
    results = []
    for name, path in generated_files:
        if path.exists():
            size_kb = path.stat().st_size / 1024
            print(f"  PASS  {name}: {size_kb:.1f} KB")
            results.append(True)
        else:
            print(f"  FAIL  {name}: Not generated")
            results.append(False)
    
    passed = sum(results)
    total = len(results)
    print(f"\n{passed}/{total} presentations generated successfully.")
    print(f"Output directory: {OUTPUT_DIR}")
    
    if passed == total:
        print("\nAll PowerPoint features working!")
    else:
        print("\nSome features need attention.")
else:
    print("PowerPoint tests skipped — python-pptx not installed.")

## Related
- Source: `siege_utilities/reporting/powerpoint_generator.py`, `siege_utilities/reporting/pages/`
- Tests: `tests/test_survey_render.py`
